# 上側・下側 CUSUM（NIST ベース）

このノートブックには、次の関数が入っています。

- `fit_cusum_params` : 正常時データから CUSUM パラメータを計算
- `compute_cusum` : 上側・下側 CUSUM を計算
- `summarize_cusum_results` : 警報結果を表に要約
- `plot_cusum_results` : 可視化
- `run_cusum_pipeline` : 一連の処理をまとめて実行

想定するデータ形式は、**行が時系列、列がセンサや特徴量**です。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _select_numeric_columns(df: pd.DataFrame, columns=None):
    """
    監視対象の列名を決める補助関数。

    Parameters
    ----------
    df : pd.DataFrame
        元のデータフレーム。
    columns : list[str] or None
        監視したい列名の一覧。
        None の場合は、df の数値列を自動で選ぶ。

    Returns
    -------
    cols : list[str]
        実際に使用する列名の一覧。
    """
    # columns が未指定なら、数値列だけ自動で対象にする
    if columns is None:
        cols = df.select_dtypes(include=[np.number]).columns.tolist()
    else:
        cols = list(columns)

    # 対象列が1つもなければエラー
    if len(cols) == 0:
        raise ValueError("監視対象の数値列が見つかりません。")

    # 指定された列名が df に存在するか確認
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"df に存在しない列があります: {missing}")

    return cols


def _to_series(values, index, name):
    """
    スカラー / dict / Series を、列名 index に対応した pandas.Series にそろえる補助関数。

    たとえば target_values や sigma_values は、
    - 全列で同じ値を使う: scalar
    - 列ごとに違う値を使う: dict や Series
    の両方を許したいので、この関数で形を統一する。

    Parameters
    ----------
    values : scalar / dict / pd.Series / None
        変換したい値。
    index : list-like
        出力 Series の index（通常は列名一覧）。
    name : str
        エラーメッセージや Series 名に使う名前。

    Returns
    -------
    s : pd.Series or None
        index にそろえた Series。
        values が None のときは None を返す。
    """
    if values is None:
        return None

    # 1つの数値なら、全列に同じ値を割り当てる
    if np.isscalar(values):
        return pd.Series(float(values), index=index, name=name, dtype=float)

    # dict や Series の場合は Series 化して index に合わせる
    s = pd.Series(values, dtype=float, name=name).reindex(index)

    # 一部の列に値が入っていなければエラー
    if s.isna().any():
        raise ValueError(f"{name} に不足している列があります: {s[s.isna()].index.tolist()}")

    return s


def fit_cusum_params(
    baseline_df: pd.DataFrame,
    columns=None,
    alpha: float = 0.0027,
    beta: float = 0.01,
    delta: float = 1.0,
    target_values=None,
    sigma_values=None,
    ddof: int = 1,
):
    """
    正常時データ（baseline_df）から、CUSUM の設計パラメータを列ごとに計算する。

    NIST の表形式 CUSUM を想定して、列ごとに以下を求める:
    - mu0     : 目標平均
    - sigma_x : 標準偏差
    - k       : 参照値（reference value）
    - d       : 設計用の中間値
    - h       : 判定限界（decision interval）

    Parameters
    ----------
    baseline_df : pd.DataFrame
        正常時データ。
        行方向に時系列、列方向にセンサや特徴量を想定。
    columns : list[str] or None
        対象列。None の場合は数値列を自動選択。
    alpha : float
        誤警報率。
    beta : float
        見逃し率。
    delta : float
        検知したい平均シフト量。
        「何σずれたら検知したいか」に対応する値。
    target_values : None / scalar / dict / pd.Series
        既知の目標平均 mu0 を使いたい場合に指定。
        None なら baseline_df の列平均を使う。
    sigma_values : None / scalar / dict / pd.Series
        既知の標準偏差 sigma_x を使いたい場合に指定。
        None なら baseline_df の列標準偏差を使う。
    ddof : int
        標準偏差計算時の自由度。
        通常は標本標準偏差として ddof=1 を使う。

    Returns
    -------
    params_df : pd.DataFrame
        列ごとの CUSUM パラメータ表。
        index は列名。
    """
    # 対象列を決める
    cols = _select_numeric_columns(baseline_df, columns)

    # 必要な列だけ抜き出す
    base = baseline_df[cols].copy()

    # baseline 側に欠損があると、平均・標準偏差の意味が不安定になるのでここではエラーにする
    if base.isna().any().any():
        raise ValueError("baseline_df に欠損値があります。先に補完または除外してください。")

    # mu0 を用意
    # ユーザー指定があればそれを使う
    # なければ baseline の列平均を使う
    mu0 = _to_series(target_values, cols, "mu0")
    if mu0 is None:
        mu0 = base.mean()

    # sigma_x を用意
    # ユーザー指定があればそれを使う
    # なければ baseline の列標準偏差を使う
    sigma_x = _to_series(sigma_values, cols, "sigma_x")
    if sigma_x is None:
        sigma_x = base.std(ddof=ddof)

    # 標準偏差が 0 以下だと CUSUM の設計ができない
    if (sigma_x <= 0).any():
        bad_cols = sigma_x[sigma_x <= 0].index.tolist()
        raise ValueError(f"sigma_x が 0 以下の列があります: {bad_cols}")

    # ---- NIST ベースの設計式 ----
    # k は「どれくらいのずれを蓄積し始めるか」の基準
    # delta=1 のとき、k = sigma/2
    k = (delta * sigma_x) / 2.0

    # d は alpha, beta, delta から決まる中間値
    d = (2.0 / (delta ** 2)) * np.log((1.0 - beta) / alpha)

    # h は実際の判定限界
    # 上側CUSUMまたは下側CUSUMが h を超えると異常
    h = d * k

    # 列ごとのパラメータ表にまとめる
    params_df = pd.DataFrame(
        {
            "mu0": mu0.astype(float),
            "sigma_x": sigma_x.astype(float),
            "alpha": float(alpha),
            "beta": float(beta),
            "delta": float(delta),
            "k": k.astype(float),
            "d": float(d),
            "h": h.astype(float),
        }
    )

    return params_df


def compute_cusum(
    monitor_df: pd.DataFrame,
    params_df: pd.DataFrame,
    columns=None,
    na_action: str = "raise",
):
    """
    既に求めた params_df を使って、上側・下側 CUSUM を列ごとに計算する。

    上側CUSUM:
        Shi(i) = max(0, Shi(i-1) + x_i - mu0 - k)

    下側CUSUM:
        Slo(i) = max(0, Slo(i-1) + mu0 - k - x_i)

    Parameters
    ----------
    monitor_df : pd.DataFrame
        監視対象データ。
        行方向が時系列、列方向がセンサや特徴量。
    params_df : pd.DataFrame
        fit_cusum_params() の出力。
    columns : list[str] or None
        対象列。None の場合は params_df.index を使う。
    na_action : {"raise", "reset"}
        monitor_df に欠損があったときの動作。
        - "raise": 欠損があればエラーにする
        - "reset": その時点のCUSUMを NaN にし、内部状態を 0 に戻して継続

    Returns
    -------
    result : dict[str, pd.DataFrame]
        以下の DataFrame をまとめた辞書:
        - x           : 元データ
        - cusum_plain : 普通の累積和（参考用）
        - cusum_hi    : 上側 CUSUM
        - cusum_lo    : 下側 CUSUM
        - alarm_hi    : 上側警報フラグ
        - alarm_lo    : 下側警報フラグ
        - alarm_any   : 上下どちらか警報
    """
    # 対象列の決定
    if columns is None:
        cols = list(params_df.index)
    else:
        cols = list(columns)

    # monitor_df に対象列があるか確認
    missing_in_monitor = [c for c in cols if c not in monitor_df.columns]
    if missing_in_monitor:
        raise KeyError(f"monitor_df に存在しない列があります: {missing_in_monitor}")

    # params_df に対象列があるか確認
    missing_in_params = [c for c in cols if c not in params_df.index]
    if missing_in_params:
        raise KeyError(f"params_df に存在しない列があります: {missing_in_params}")

    # 必要列だけ取り出す
    x_df = monitor_df[cols].copy()

    # 欠損値処理方法の妥当性チェック
    if na_action not in {"raise", "reset"}:
        raise ValueError("na_action は 'raise' または 'reset' を指定してください。")

    # 欠損値を許さない設定なら、先にまとめてチェック
    if na_action == "raise" and x_df.isna().any().any():
        raise ValueError("monitor_df に欠損値があります。na_action='reset' を使うか、先に処理してください。")

    # 出力用の空 DataFrame を作る
    idx = x_df.index
    cusum_plain = pd.DataFrame(index=idx, columns=cols, dtype=float)
    cusum_hi = pd.DataFrame(index=idx, columns=cols, dtype=float)
    cusum_lo = pd.DataFrame(index=idx, columns=cols, dtype=float)
    alarm_hi = pd.DataFrame(index=idx, columns=cols, dtype=bool)
    alarm_lo = pd.DataFrame(index=idx, columns=cols, dtype=bool)
    alarm_any = pd.DataFrame(index=idx, columns=cols, dtype=bool)

    # ---- 列ごとに独立に CUSUM を計算 ----
    for col in cols:
        # その列の観測値
        x = x_df[col].to_numpy(dtype=float)

        # その列専用の設計パラメータ
        mu0 = float(params_df.loc[col, "mu0"])
        k = float(params_df.loc[col, "k"])
        h = float(params_df.loc[col, "h"])

        # 結果格納用配列
        plain_arr = np.full(len(x), np.nan, dtype=float)
        hi_arr = np.full(len(x), np.nan, dtype=float)
        lo_arr = np.full(len(x), np.nan, dtype=float)

        # 1つ前の時点の内部状態
        prev_plain = 0.0
        prev_hi = 0.0
        prev_lo = 0.0

        # ---- 時系列順に 1 点ずつ更新 ----
        for i, xi in enumerate(x):
            # 欠損値なら設定に応じて処理
            if np.isnan(xi):
                if na_action == "raise":
                    raise ValueError(f"{col} に欠損値があります。")

                # reset の場合:
                # その位置の出力は NaN にし、
                # 次の時点からは CUSUM を 0 から再開する
                plain_arr[i] = np.nan
                hi_arr[i] = np.nan
                lo_arr[i] = np.nan
                prev_plain = 0.0
                prev_hi = 0.0
                prev_lo = 0.0
                continue

            # 普通の累積和（参考表示用）
            # これは異常判定には直接使わない
            prev_plain = prev_plain + (xi - mu0)

            # 上側 CUSUM
            # mu0 より高い方向のずれが続くと増えていく
            # 少し戻ったときは max(0, ...) で 0 までリセットされる
            prev_hi = max(0.0, prev_hi + xi - mu0 - k)

            # 下側 CUSUM
            # mu0 より低い方向のずれが続くと増えていく
            prev_lo = max(0.0, prev_lo + mu0 - k - xi)

            # 配列に保存
            plain_arr[i] = prev_plain
            hi_arr[i] = prev_hi
            lo_arr[i] = prev_lo

        # 列単位の結果を DataFrame に格納
        cusum_plain[col] = plain_arr
        cusum_hi[col] = hi_arr
        cusum_lo[col] = lo_arr

        # h を超えた時点を警報とする
        alarm_hi[col] = hi_arr > h
        alarm_lo[col] = lo_arr > h
        alarm_any[col] = (hi_arr > h) | (lo_arr > h)

    return {
        "x": x_df,
        "cusum_plain": cusum_plain,
        "cusum_hi": cusum_hi,
        "cusum_lo": cusum_lo,
        "alarm_hi": alarm_hi,
        "alarm_lo": alarm_lo,
        "alarm_any": alarm_any,
    }


def summarize_cusum_results(results: dict, params_df: pd.DataFrame):
    """
    CUSUM 計算結果を、列ごとの要約表にする。

    何回アラームが出たか、
    最初にいつアラームが出たか、
    などを一覧化する。

    Parameters
    ----------
    results : dict
        compute_cusum() の出力。
    params_df : pd.DataFrame
        fit_cusum_params() の出力。

    Returns
    -------
    summary_df : pd.DataFrame
        列ごとの要約表。
    """
    cols = list(results["x"].columns)
    rows = []

    for col in cols:
        # 上側・下側・総合の警報フラグ
        ah = results["alarm_hi"][col]
        al = results["alarm_lo"][col]
        aa = results["alarm_any"][col]

        # 最初に警報が出た index を求める
        # 出ていなければ pd.NA
        first_hi = ah.index[ah].tolist()[0] if ah.any() else pd.NA
        first_lo = al.index[al].tolist()[0] if al.any() else pd.NA
        first_any = aa.index[aa].tolist()[0] if aa.any() else pd.NA

        rows.append(
            {
                "column": col,
                "mu0": params_df.loc[col, "mu0"],
                "sigma_x": params_df.loc[col, "sigma_x"],
                "k": params_df.loc[col, "k"],
                "h": params_df.loc[col, "h"],
                "n_alarm_hi": int(ah.sum()),
                "n_alarm_lo": int(al.sum()),
                "n_alarm_any": int(aa.sum()),
                "first_alarm_hi": first_hi,
                "first_alarm_lo": first_lo,
                "first_alarm_any": first_any,
            }
        )

    return pd.DataFrame(rows).set_index("column")


def plot_cusum_results(
    results: dict,
    params_df: pd.DataFrame,
    columns=None,
    figsize=None,
    show_plain_cusum: bool = True,
):
    """
    CUSUM の計算結果を可視化する。

    各列について2段で表示:
    1段目: 元データ x
    2段目: 上側CUSUM, 下側CUSUM

    グラフ上では下側CUSUMは見やすさのため負方向に描く。
    つまり内部計算では正の値で保持しているが、
    描画では -Slo として表示する。

    Parameters
    ----------
    results : dict
        compute_cusum() の出力。
    params_df : pd.DataFrame
        fit_cusum_params() の出力。
    columns : list[str] or None
        描画対象列。None の場合は results["x"] の全列。
    figsize : tuple or None
        図サイズ。None の場合は自動設定。
    show_plain_cusum : bool
        参考用の普通の累積和も描くかどうか。

    Returns
    -------
    fig, axes
        matplotlib の Figure と Axes。
    """
    # 描画対象列を決める
    if columns is None:
        cols = list(results["x"].columns)
    else:
        cols = list(columns)

    n = len(cols)
    if n == 0:
        raise ValueError("可視化対象の列がありません。")

    # 列数に応じて図の大きさを自動設定
    if figsize is None:
        figsize = (12, max(5, 4 * n))

    # 1列につき2段のグラフを用意
    fig, axes = plt.subplots(
        nrows=2 * n,
        ncols=1,
        figsize=figsize,
        sharex=True,
        constrained_layout=True,
    )

    # n=1 のとき axes が1本になる場合に備えて配列化
    if 2 * n == 1:
        axes = np.array([axes])

    for i, col in enumerate(cols):
        # 上段と下段の軸
        ax_top = axes[2 * i]
        ax_bottom = axes[2 * i + 1]

        # 描画に使うデータ
        x = results["x"][col]
        hi = results["cusum_hi"][col]
        lo = results["cusum_lo"][col]
        plain = results["cusum_plain"][col]
        ah = results["alarm_hi"][col]
        al = results["alarm_lo"][col]
        aa = results["alarm_any"][col]

        # その列の設計値
        mu0 = float(params_df.loc[col, "mu0"])
        k = float(params_df.loc[col, "k"])
        h = float(params_df.loc[col, "h"])

        # ===== 上段: 元データ =====
        # 観測値
        ax_top.plot(x.index, x.values, marker="o", linewidth=1.2, label="x")

        # 目標平均
        ax_top.axhline(mu0, linestyle="--", linewidth=1.2, label="mu0")

        # mu0 ± k も補助線として表示
        # どのくらいずれたら累積しやすくなるかの目安
        ax_top.axhline(mu0 + k, linestyle=":", linewidth=1.0, label="mu0 + k")
        ax_top.axhline(mu0 - k, linestyle=":", linewidth=1.0, label="mu0 - k")

        # どこかで異常判定が出た観測点に印を付ける
        if aa.any():
            ax_top.scatter(
                x.index[aa],
                x[aa],
                marker="x",
                s=60,
                label="alarm point",
            )

        ax_top.set_title(f"{col} : monitored values")
        ax_top.set_ylabel("value")
        ax_top.legend(loc="best")

        # ===== 下段: CUSUM =====
        # 上側 CUSUM はそのまま上に描く
        ax_bottom.plot(hi.index, hi.values, linewidth=1.5, label="Upper CUSUM (Shi)")

        # 下側 CUSUM は見やすさのため符号を反転して下に描く
        ax_bottom.plot(lo.index, -lo.values, linewidth=1.5, label="Lower CUSUM (-Slo)")

        # 閾値 ±h
        ax_bottom.axhline(h, linestyle="--", linewidth=1.2, label="+h")
        ax_bottom.axhline(-h, linestyle="--", linewidth=1.2, label="-h")

        # 参考用の通常の累積和も必要なら描く
        if show_plain_cusum:
            ax_bottom.plot(
                plain.index,
                plain.values,
                linestyle=":",
                linewidth=1.0,
                label="Plain CUSUM",
            )

        # 上側警報点
        if ah.any():
            ax_bottom.scatter(
                hi.index[ah],
                hi[ah],
                marker="^",
                s=60,
                label="upper alarm",
            )

        # 下側警報点
        # 描画は -lo 側に置く
        if al.any():
            ax_bottom.scatter(
                lo.index[al],
                -lo[al],
                marker="v",
                s=60,
                label="lower alarm",
            )

        ax_bottom.set_title(f"{col} : tabular CUSUM")
        ax_bottom.set_ylabel("CUSUM")
        ax_bottom.legend(loc="best")

    axes[-1].set_xlabel("index")
    return fig, axes


def run_cusum_pipeline(
    baseline_df: pd.DataFrame,
    monitor_df: pd.DataFrame = None,
    columns=None,
    alpha: float = 0.0027,
    beta: float = 0.01,
    delta: float = 1.0,
    target_values=None,
    sigma_values=None,
    ddof: int = 1,
    na_action: str = "raise",
    figsize=None,
    show_plain_cusum: bool = True,
):
    """
    CUSUM の一連の処理をまとめて実行する関数。

    処理の流れ:
    1. baseline_df から設計パラメータを作る
    2. monitor_df に対して上側・下側 CUSUM を計算する
    3. 計算結果を要約表にする
    4. グラフで可視化する

    Parameters
    ----------
    baseline_df : pd.DataFrame
        正常時データ。
        CUSUM の基準値 mu0 や sigma_x を求めるために使う。
    monitor_df : pd.DataFrame or None
        監視対象データ。
        None の場合は baseline_df 自身を監視対象として使う。
    columns : list[str] or None
        対象列。None なら自動選択。
    alpha, beta, delta : float
        NIST ベースの設計パラメータ。
    target_values : None / scalar / dict / pd.Series
        mu0 を直接指定したい場合に使う。
    sigma_values : None / scalar / dict / pd.Series
        sigma_x を直接指定したい場合に使う。
    ddof : int
        標準偏差計算の自由度。
    na_action : {"raise", "reset"}
        欠損値への対応。
    figsize : tuple or None
        描画サイズ。
    show_plain_cusum : bool
        普通の累積和も表示するかどうか。

    Returns
    -------
    params_df : pd.DataFrame
        設計パラメータ表。
    results : dict
        CUSUM 計算結果一式。
    summary_df : pd.DataFrame
        警報要約表。
    fig, axes
        matplotlib の Figure と Axes。
    """
    # monitor_df が省略されたら、baseline 自身を監視対象にする
    if monitor_df is None:
        monitor_df = baseline_df.copy()

    # 1. 正常時データから CUSUM パラメータを作る
    params_df = fit_cusum_params(
        baseline_df=baseline_df,
        columns=columns,
        alpha=alpha,
        beta=beta,
        delta=delta,
        target_values=target_values,
        sigma_values=sigma_values,
        ddof=ddof,
    )

    # 2. 監視対象データに対して CUSUM を計算する
    results = compute_cusum(
        monitor_df=monitor_df,
        params_df=params_df,
        columns=columns,
        na_action=na_action,
    )

    # 3. 列ごとの警報結果を表にまとめる
    summary_df = summarize_cusum_results(results, params_df)

    # 4. 可視化
    fig, axes = plot_cusum_results(
        results=results,
        params_df=params_df,
        columns=columns,
        figsize=figsize,
        show_plain_cusum=show_plain_cusum,
    )

    return params_df, results, summary_df, fig, axes


In [ ]:
# =========================
# 使用例
# =========================

# 例として、baseline_df は正常時データ、
# monitor_df は監視したい時系列データを想定する
#
# 行:
#   時系列順の観測点
# 列:
#   センサ値や特徴量

# baseline_df = ...
# monitor_df = ...

params_df, results, summary_df, fig, axes = run_cusum_pipeline(
    baseline_df=baseline_df,
    monitor_df=monitor_df,
    columns=["sensor_1", "sensor_2", "sensor_3"],  # None にすると数値列を自動選択
    alpha=0.0027,   # 誤警報率
    beta=0.01,      # 見逃し率
    delta=1.0,      # 1σ の平均シフトを検知したい設定
    na_action="raise",
)

print("=== CUSUM パラメータ ===")
print(params_df)

print("\n=== 警報要約 ===")
print(summary_df)

plt.show()
